In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder


Activation functions

In [2]:
def sigmoid(Z):
    return 1 / (1 + np.exp(-Z))

def relu(Z):
    return np.maximum(0, Z)

def relu_backward(dA, Z):
    dZ = np.array(dA, copy=True)
    dZ[Z <= 0] = 0
    return dZ

---------------- Forward propagation ----------------

In [3]:
def forward_propagation(X, parameters):
    caches = {}
    A = X
    caches["A0"] = A
    L = len(parameters) // 2

    for l in range(1, L):
        W = parameters["W" + str(l)]
        b = parameters["b" + str(l)]
        Z = np.dot(W, A) + b
        A = relu(Z)
        caches["Z" + str(l)] = Z
        caches["A" + str(l)] = A

    WL = parameters["W" + str(L)]
    bL = parameters["b" + str(L)]
    ZL = np.dot(WL, A) + bL
    AL = sigmoid(ZL)

    caches["Z" + str(L)] = ZL
    caches["A" + str(L)] = AL

    return AL, caches

---------------- Backward propagation ----------------

In [4]:
def backward_propagation(X, Y, parameters, caches):
    grads = {}
    m = X.shape[1]
    L = len(parameters) // 2

    AL = caches["A" + str(L)]

    dZL = AL - Y
    grads["dW" + str(L)] = (1/m) * np.dot(dZL, caches["A" + str(L-1)].T)
    grads["db" + str(L)] = (1/m) * np.sum(dZL, axis=1, keepdims=True)

    dA_prev = np.dot(parameters["W" + str(L)].T, dZL)

    for l in reversed(range(1, L)):
        dZ = relu_backward(dA_prev, caches["Z" + str(l)])
        grads["dW" + str(l)] = (1/m) * np.dot(dZ, caches["A" + str(l-1)].T)
        grads["db" + str(l)] = (1/m) * np.sum(dZ, axis=1, keepdims=True)
        dA_prev = np.dot(parameters["W" + str(l)].T, dZ)

    return grads

---------------- Update parameters ----------------

In [5]:
def update_parameters(parameters, grads, lr=0.05):
    L = len(parameters) // 2

    for l in range(1, L+1):
        parameters["W" + str(l)] -= lr * grads["dW" + str(l)]
        parameters["b" + str(l)] -= lr * grads["db" + str(l)]

    return parameters

---------------- Loss ----------------

In [6]:
def compute_loss(AL, Y):
    m = Y.shape[1]
    loss = -(1/m) * np.sum(
        Y*np.log(AL + 1e-8) + (1-Y)*np.log(1-AL + 1e-8)
    )
    return loss

In [7]:
data = pd.read_csv("Teen_Mental_Health_Dataset.csv")

---------------- Encode gender (لو نص) ----------------

In [8]:
le = LabelEncoder()
data["gender"] = le.fit_transform(data["gender"])
data["platform_usage"] = le.fit_transform(data["platform_usage"])



In [9]:
data.fillna(data.mean(numeric_only=True), inplace=True)

In [10]:
level_map = {
    "low": 0,
    "medium": 1,
    "high": 2
}

cols_to_map = [
    "academic_performance",
    "physical_activity",
    "social_interaction_level",
    "stress_level",
    "anxiety_level",
    "addiction_level"
]

for col in cols_to_map:
    data[col] = data[col].map(level_map)

---------------- Split ----------------

In [11]:

X = data.drop("depression_label", axis=1).values
Y = data["depression_label"].values


 #---------------- Prepare ----------------

In [12]:
X = X.astype(float)
X = X.T
Y = Y.reshape(1, -1)

---------------- Scaling ----------------

In [13]:
mean = np.mean(X, axis=1, keepdims=True)
std = np.std(X, axis=1, keepdims=True) + 1e-8
X = (X - mean) / std

---------------- Initialize ----------------

In [14]:
np.random.seed(1)

parameters = {
    "W1": np.random.randn(10, X.shape[0]) * 0.01,
    "b1": np.zeros((10,1)),
    "W2": np.random.randn(6, 10) * 0.01,
    "b2": np.zeros((6,1)),
    "W3": np.random.randn(1, 6) * 0.01,
    "b3": np.zeros((1,1))
}


 ---------------- Training ----------------

In [15]:
for i in range(5000):
    AL, caches = forward_propagation(X, parameters)
    loss = compute_loss(AL, Y)
    grads = backward_propagation(X, Y, parameters, caches)
    parameters = update_parameters(parameters, grads)

    if i % 500 == 0:
        print(f"Iteration {i} | Loss: {loss:.4f}")

Iteration 0 | Loss: nan
Iteration 500 | Loss: nan
Iteration 1000 | Loss: nan
Iteration 1500 | Loss: nan
Iteration 2000 | Loss: nan
Iteration 2500 | Loss: nan
Iteration 3000 | Loss: nan
Iteration 3500 | Loss: nan
Iteration 4000 | Loss: nan
Iteration 4500 | Loss: nan


---------------- Prediction ----------------

In [16]:
AL, _ = forward_propagation(X, parameters)
preds = (AL > 0.5).astype(int)

print("\nPredictions:\n", preds)


Predictions:
 [[0 0 0 ... 0 0 0]]
